In [ ]:
import os
import mlflow
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score
from sklearn.model_selection import train_test_split

import sys
sys.path.insert(0, "../../run")
from run_config import REPO_PATH, SEASONS, PROCESSED_DATA_PATH, FEATURES_PATH
sys.path.insert(1, f"{REPO_PATH}")
from experiment_config import *

from src.model.models import get_rf_model

In [ ]:
features_path = f"{FEATURES_PATH}/all_combined_features.csv"
output_dir = f"{REPO_PATH}/models/ml_models"

In [3]:
seasons=sorted(SEASONS)
target_dfs=[pd.read_csv(f"{PROCESSED_DATA_PATH}/{season}/all_target_df.csv") for season in seasons[1:]]
for df in target_dfs:
    df['date']= pd.to_datetime(df['date'])
target_df = pd.concat(target_dfs, ignore_index=True)
target_columns=target_df.drop(columns=key_columns).columns.tolist()

In [5]:
TARGET_RANGES={
    'home_goals': [0,5],
    'away_goals': [0,6],
    'home_corners': [3,7],
    'away_corners': [3,7],
    'home_cards': [0, 7],
    'away_cards': [0, 7],
    'home_shots': [5, 20],
    'away_shots': [5, 20],
    'home_sots': [1, 10],
    'away_sots':[1, 10],
}

In [6]:
test_size = 0.2
random_state = 42

experiment_name = "mlflow_test"

model_params={
    "n_estimators": 100,
    "max_depth": 10,
    "random_state": random_state
}

In [7]:
def align_on_keys(feature_df: pd.DataFrame, target_df: pd.DataFrame, key_columns: list[str]):
    """Aligns feature and target DataFrames on key columns, and checks row-wise key alignment."""
    # Check keys exist
    for df_name, df in [('features', feature_df), ('targets', target_df)]:
        missing_keys = [k for k in key_columns if k not in df.columns]
        if missing_keys:
            raise ValueError(f"Missing keys {missing_keys} in {df_name} DataFrame")

    # Merge on key columns
    merged = pd.merge(feature_df, target_df, on=key_columns, how='inner', suffixes=("", "_target"))
    
    if merged.empty:
        raise ValueError("No matching rows found on key columns. Check for mismatches in 'home', 'away', 'date'.")

    # Double-check alignment by comparing key columns row-wise
    for key in key_columns:
        if not (merged[key] == merged[f"{key}"]).all():
            raise ValueError(f"Mismatch detected in key column '{key}' after merging.")

    # Separate aligned outputs
    X_aligned = merged[feature_df.columns]
    target_cols = [col for col in target_df.columns if col not in key_columns]
    target_aligned = merged[target_cols]

    return X_aligned.reset_index(drop=True), target_aligned.reset_index(drop=True)


In [8]:
def log_threshold_metrics(y_true, y_pred, lower, upper, target_name):
    for i in range(lower, upper):
        y_true_bin = (y_true > i).astype(int)
        y_pred_bin = (y_pred > i).astype(int)

        acc = accuracy_score(y_true_bin, y_pred_bin)
        prec = precision_score(y_true_bin, y_pred_bin, zero_division=0)

        mlflow.log_metric(f"{target_name}_gt_{i}_accuracy", acc)
        mlflow.log_metric(f"{target_name}_gt_{i}_precision", prec)

In [9]:
def train_and_log_model(X, y, target_name, model_params, output_dir, target_ranges):

    lower, upper = target_ranges[target_name]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=None
    )

    model = get_rf_model(model_params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Log model
    model_path = os.path.join(output_dir, f"{target_name}.joblib")
    os.makedirs(output_dir, exist_ok=True)
    joblib.dump(model, model_path)

    mlflow.sklearn.log_model(
        model,
        artifact_path=f"model_{target_name}",
        input_example=X_test.iloc[:5]  # Small sample of input features
    )
    mlflow.log_param(f"model_{target_name}", model.__class__.__name__)
    mlflow.log_param(f"{target_name}_range", f"{lower}-{upper}")

    # Log threshold-based metrics
    log_threshold_metrics(y_test, y_pred, lower, upper, target_name)
    print(f"Logged model for {target_name}")

In [10]:
seasons=sorted(SEASONS)
target_dfs=[pd.read_csv(f"{PROCESSED_DATA_PATH}/{season}/all_target_df.csv") for season in seasons]

In [11]:
for df in target_dfs:
    df['date']= pd.to_datetime(df['date'])

In [12]:
mlflow.set_tracking_uri(f'{REPO_PATH}/mlflow/')
mlflow.set_experiment(experiment_name)
feature_df = pd.read_csv(features_path)
feature_df['date']= pd.to_datetime(feature_df['date'])
feature_df, target_df=align_on_keys(feature_df, target_df, key_columns)
with mlflow.start_run():
    for target in target_columns:
        if target not in target_df.columns:
            print(f"Warning: Target '{target}' not found in target file, skipping.")
            continue

        y = target_df[target]
        train_and_log_model(feature_df.drop(columns=key_columns), y, target, model_params, output_dir, TARGET_RANGES)

2025/06/14 23:20:05 INFO mlflow.tracking.fluent: Experiment with name 'mlflow_test' does not exist. Creating a new experiment.
2025/06/14 23:20:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
 

Logged model for home_goals


2025/06/14 23:20:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Logged model for away_goals


2025/06/14 23:20:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


...

Logged model for away_shots


2025/06/14 23:20:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Logged model for home_sots


2025/06/14 23:20:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Logged model for away_sots
